# 03 - Visualization (Member 1)
Class distribution, missing values, text length, word count and engagement distributions (where available).
Figures are saved to `results/figures/`.

In [ ]:
import sys, os
from pathlib import Path
# --- Colab: uncomment to mount Drive (datasets stay OUT of GitHub) ---
# from google.colab import drive; drive.mount('/content/drive')
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
FIG = Path("results/figures"); FIG.mkdir(parents=True, exist_ok=True)
NAMES = ["fakehealth", "coaid", "pubhealth"]
raw = {n: pd.read_csv(f"data/processed/{n}_raw_standardized.csv") for n in NAMES if Path(f"data/processed/{n}_raw_standardized.csv").exists()}
proc = {n: pd.read_csv(f"data/processed/{n}_processed.csv") for n in NAMES if Path(f"data/processed/{n}_processed.csv").exists()}
print("raw:", list(raw), "| processed:", list(proc))

## 1. Class distribution (processed)

In [ ]:
fig, axes = plt.subplots(1, len(proc), figsize=(5*len(proc), 4), squeeze=False)
for ax, (n, df) in zip(axes[0], proc.items()):
    sns.countplot(x=df["label"].map({0: "real", 1: "fake"}), order=["real", "fake"], ax=ax)
    ax.set_title(n); ax.set_xlabel("")
    for p in ax.patches: ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(FIG/"class_distribution.png", dpi=150); plt.show()

## 2. Missing values (raw standardized)

In [ ]:
fig, axes = plt.subplots(1, len(raw), figsize=(5*len(raw), 4), squeeze=False)
for ax, (n, df) in zip(axes[0], raw.items()):
    (df.isna().mean()*100).sort_values().plot.barh(ax=ax); ax.set_title(n); ax.set_xlabel("% missing")
plt.tight_layout(); plt.savefig(FIG/"missing_values.png", dpi=150); plt.show()

## 3. Text length and word count by class (processed)

In [ ]:
for col, fname in (("text_len", "text_length"), ("word_count", "word_count")):
    fig, axes = plt.subplots(1, len(proc), figsize=(5*len(proc), 4), squeeze=False)
    for ax, (n, df) in zip(axes[0], proc.items()):
        hi = df[col].quantile(0.99)   # clip extreme tail for readability only
        sns.histplot(data=df[df[col] <= hi], x=col, hue=df["label"].map({0: "real", 1: "fake"}), bins=40, ax=ax)
        ax.set_title(f"{n} - {col} (<=99th pct)")
    plt.tight_layout(); plt.savefig(FIG/f"{fname}.png", dpi=150); plt.show()

In [ ]:
print(pd.concat({n: d.groupby("label")[["text_len","word_count"]].describe().T for n, d in proc.items()}, axis=1))

## 4. Engagement distributions (only datasets where available)

In [ ]:
eng_cols = ["n_tweets", "n_replies", "n_retweets"]
avail = {n: d for n, d in proc.items() if d["has_engagement"].any()}
if not avail:
    print("No engagement data available in the processed datasets.")
for n, d in avail.items():
    d = d[d["has_engagement"]]
    cols = [c for c in eng_cols if d[c].notna().any()]
    fig, axes = plt.subplots(1, len(cols), figsize=(5*len(cols), 4), squeeze=False)
    for ax, c in zip(axes[0], cols):
        sns.histplot(np.log1p(d[c].dropna()), bins=40, ax=ax); ax.set_xlabel(f"log(1+{c})"); ax.set_title(f"{n}: {c}")
    plt.tight_layout(); plt.savefig(FIG/f"engagement_{n}.png", dpi=150); plt.show()

## 5. Context availability per dataset (motivates the missing-context experiments)

In [ ]:
avail_tbl = pd.DataFrame({n: d[["has_source","has_engagement","has_network"]].mean() for n, d in proc.items()}).T
display(avail_tbl.round(3))
avail_tbl.plot.bar(figsize=(6,4), ylim=(0,1)); plt.ylabel("fraction of samples"); plt.tight_layout()
plt.savefig(FIG/"context_availability.png", dpi=150); plt.show()